In [ ]:
import os
import ast
from typing import List

import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb
import matplotlib.pyplot as plt

In [ ]:

# =========================
# Configuration
# =========================

DATASET_URL = "FronkonGames/steam-games-dataset"
DATASET_SPLIT = "train"
LOCAL_CLEANED_FILE = "steam_games_cleaned.csv"
LOCAL_FEATURE_FILE = "steam_games_feature.csv"
RANDOM_STATE = 42


In [ ]:
# =========================
# 1. Data Loading
# =========================

def load_steam_dataset() -> pd.DataFrame:
    print(f"Attempting to load dataset from Hugging Face: {DATASET_URL} ...")

    try:
        dataset_dict = load_dataset(DATASET_URL)
        split = DATASET_SPLIT if DATASET_SPLIT in dataset_dict else list(dataset_dict.keys())[0]
        df = dataset_dict[split].to_pandas()
        print(f"Dataset loaded successfully from split: '{split}'")
        print(f"Raw data shape: {df.shape[0]} rows x {df.shape[1]} columns")
        return df

    except Exception as e:
        print(f"Error loading dataset from Hugging Face: {e}")

        if os.path.exists(LOCAL_CLEANED_FILE):
            print(f"Fallback: loading existing cleaned file: {LOCAL_CLEANED_FILE}")
            df = pd.read_csv(LOCAL_CLEANED_FILE)
            print(f"Loaded local cleaned data: {df.shape[0]} rows x{df.shape[1]} columns")
            return df

        raise SystemExit("Cannot continue: dataset not found on HuggingFace and no local backup file.")




In [ ]:
# =========================
# 2. Basic EDA
# =========================

def basic_eda(df: pd.DataFrame) -> None:
    print("\n=== Basic EDA ===")
    print(f"Initial Shape: {df.shape}")

    print("\nDataFrame info:")
    df.info()

    print("\nHead (first 5 rows):")
    print(df.head())

    print("\nMissing values per column:")
    print(df.isnull().sum())

    print("\nNumber of unique values per column:")
    print(df.nunique())





In [ ]:
# =========================
# 3. Data Cleaning
# =========================


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    print("\n=== Data Cleaning ===")

    # 只保留核心欄位
    core_columns: List[str] = [
        "Name", "Release date", "Estimated owners", "Peak CCU", "Required age", "Price",
        "DLC count", "Supported languages", "Full audio languages",
        "Windows", "Mac", "Linux",
        "Metacritic score", "User score", "Positive", "Negative", "Achievements",
        "Recommendations", "Average playtime forever", "Average playtime two weeks",
        "Median playtime forever", "Median playtime two weeks",
        "Developers", "Publishers", "Categories", "Genres",
    ]

    existing_cols = [c for c in core_columns if c in df.columns]
    df = df[existing_cols].copy()
    print(f"保留核心欄位後 shape: {df.shape}")

    # 處理 Estimated owners → owners_mid（區間中間值）
    def parse_owners(x):
        try:
            s = str(x)
            parts = s.replace(",", "").split(" - ")
            low = int(parts[0])
            high = int(parts[-1]) if len(parts) > 1 else low
            return (low + high) / 2
        except Exception:
            return np.nan

    if "Estimated owners" in df.columns:
        df["owners_mid"] = df["Estimated owners"].apply(parse_owners)
        print("Parsed 'Estimated owners' → 'owners_mid'")
    else:
        print("Column 'Estimated owners' not found; 'owners_mid' will be NaN.")
        df["owners_mid"] = np.nan

    # 將 Release date 轉成 datetime
    if "Release date" in df.columns:
        df["Release date"] = pd.to_datetime(df["Release date"], errors="coerce")
        print("Converted 'Release date' to datetime")

    # 數值欄位轉型（部分可能不存在，需先檢查）
    numeric_cols_to_cast = [
        "Peak CCU", "Required age", "DLC count",
        "Metacritic score", "User score",
        "Positive", "Negative",
        "Achievements", "Recommendations",
        "Average playtime forever", "Average playtime two weeks",
        "Median playtime forever", "Median playtime two weeks",
        "Price",
    ]
    for col in numeric_cols_to_cast:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 平台欄位轉為 0/1
    for col in ["Windows", "Mac", "Linux"]:
        if col in df.columns:
            df[col] = df[col].astype(float)

    # 移除 owners_mid 缺失太嚴重的列（無法當作目標值）
    before_drop = len(df)
    df = df.dropna(subset=["owners_mid"])
    after_drop = len(df)
    print(f"移除 owners_mid 為 NaN 的列: {before_drop - after_drop} rows dropped")

    # 移除遊戲名稱重複
    if "Name" in df.columns:
        before_dup = len(df)
        df = df.drop_duplicates(subset=["Name"])
        after_dup = len(df)
        print(f"移除重複遊戲名稱: {before_dup - after_dup} rows dropped")

    print(f"清理後 shape: {df.shape}")
    return df






In [ ]:
# =========================
# 4. Feature Engineering
# =========================


def count_audio_languages(val) -> int:
    if pd.isna(val):
        return 0
    s = str(val)
    # 如果是 list 的字串表示（例如 "['English', 'Japanese']"），試著用 ast 解析
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple, set)):
            return len(parsed)
    except Exception:
        pass

    # fallback: 以逗號切
    return len([x for x in s.split(",") if x.strip()])




In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    print("\n=== Feature Engineering ===")

    # 好評率
    if {"Positive", "Negative"}.issubset(df.columns):
        total_reviews = df["Positive"] + df["Negative"]
        df["positive_rate"] = df["Positive"] / total_reviews.replace(0, np.nan)
        df["positive_rate"] = df["positive_rate"].fillna(0)
    else:
        df["positive_rate"] = np.nan

    # 平台支援數
    platform_cols = [c for c in ["Windows", "Mac", "Linux"] if c in df.columns]
    if platform_cols:
        df["platform_count"] = df[platform_cols].sum(axis=1)
    else:
        df["platform_count"] = 0.0

    # 支援語言數量
    if "Supported languages" in df.columns:
        df["lang_count"] = df["Supported languages"].apply(
            lambda x: len(str(x).split(",")) if pd.notnull(x) else 0
        )
    else:
        df["lang_count"] = 0

    # 語音語言數量
    if "Full audio languages" in df.columns:
        df["audio_lang_count"] = df["Full audio languages"].apply(count_audio_languages)
    else:
        df["audio_lang_count"] = 0

    # 語音比例
    df["audio_lang_ratio"] = df["audio_lang_count"] / (df["lang_count"] + 1)

    # 近期 vs 總遊玩時數比
    if {"Average playtime two weeks", "Average playtime forever"}.issubset(df.columns):
        df["playtime_ratio_recent"] = (
            df["Average playtime two weeks"] / (df["Average playtime forever"] + 1)
        )
    else:
        df["playtime_ratio_recent"] = np.nan

    # 評價落差（玩家 vs 媒體）
    if {"User score", "Metacritic score"}.issubset(df.columns):
        df["rating_gap"] = df["User score"] - df["Metacritic score"]
    else:
        df["rating_gap"] = np.nan

    # 發售年份/年代/新舊標記
    if "Release date" in df.columns:
        df["release_year"] = df["Release date"].dt.year
    else:
        df["release_year"] = np.nan

    df["release_decade"] = (df["release_year"] // 10) * 10
    df["is_recent"] = (df["release_year"] >= 2020).astype(int)

    # 儲存清理後/特徵後的檔案（方便之後重複使用）
    df.to_csv(LOCAL_FEATURE_FILE, index=False, encoding="utf-8-sig")
    print(f"Feature engineering finished. Saved to '{LOCAL_FEATURE_FILE}'")
    print(f"Final feature dataframe shape: {df.shape}")

    return df




In [ ]:
# =========================
# 5. Feature Selection
# =========================

def run_feature_selection(df: pd.DataFrame, target_col: str = "owners_mid", top_n_plot: int = 20) -> pd.DataFrame:

    print("\n=== Feature Selection (Cross-comparison) ===")

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataframe.")

    # 只取數值欄位
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c != target_col]

    if not feature_cols:
        raise ValueError("No numeric feature columns found for feature selection.")

    X = df[feature_cols].fillna(0)
    y = df[target_col].values

    print(f"使用數值特徵數量: {len(feature_cols)}")
    print("Feature columns:")
    print(feature_cols)

    # ---------- 1. Pearson correlation ----------
    print("\n[1] Pearson correlation with target")
    corr_series = df[feature_cols + [target_col]].corr()[target_col].drop(target_col)
    corr_abs = corr_series.abs()  # 用絕對值表示強度（正負不重要，只看強度）

    # ---------- 2. Mutual Information ----------
    print("\n[2] Mutual Information with target")
    mi = mutual_info_regression(X, y, random_state=RANDOM_STATE)
    mi_series = pd.Series(mi, index=feature_cols)

    # ---------- 3. RandomForest feature importance ----------
    print("\n[3] RandomForestRegressor feature importance")
    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    rf.fit(X, y)
    rf_importances = pd.Series(rf.feature_importances_, index=feature_cols)

    # ---------- 合併成一張表 ----------
    results = pd.DataFrame({
        "corr": corr_series,             # 原始相關係數（有正負號）
        "corr_abs": corr_abs,           # 絕對值強度
        "mi": mi_series,
        "rf_importance": rf_importances,
    })

    # ---------- 正規化每一個指標到 0-1 ----------
    def min_max_norm(s: pd.Series) -> pd.Series:
        vmin = s.min()
        vmax = s.max()
        if vmax > vmin:
            return (s - vmin) / (vmax - vmin)
        else:
            # 所有值都一樣時，直接給 0
            return pd.Series(0.0, index=s.index)

    results["corr_abs_norm"] = min_max_norm(results["corr_abs"])
    results["mi_norm"] = min_max_norm(results["mi"])
    results["rf_importance_norm"] = min_max_norm(results["rf_importance"])

    # 綜合三種方法，取平均分數當作「交叉比對後的重要度」
    results["avg_score"] = results[["corr_abs_norm", "mi_norm", "rf_importance_norm"]].mean(axis=1)

    # 依照綜合分數由大到小排序
    results_sorted = results.sort_values("avg_score", ascending=False)

    # ---------- 印出完整比較表（所有特徵） ----------
    print("\n=== Feature comparison table (all features, sorted by avg_score) ===")
    print(
        results_sorted[
            ["corr", "corr_abs", "mi", "rf_importance",
             "corr_abs_norm", "mi_norm", "rf_importance_norm", "avg_score"]
        ]
    )

    # ---------- 畫比對圖（預設畫前 top_n_plot 個特徵） ----------
    top_n_plot = min(top_n_plot, len(results_sorted))
    top = results_sorted.head(top_n_plot)

    indices = np.arange(len(top))
    width = 0.25

    plt.figure(figsize=(max(10, top_n_plot * 0.5), 6))

    plt.bar(indices - width, top["corr_abs_norm"], width, label="|corr| (norm)")
    plt.bar(indices,         top["mi_norm"],       width, label="MI (norm)")
    plt.bar(indices + width, top["rf_importance_norm"], width, label="RF importance (norm)")

    plt.xticks(indices, top.index, rotation=45, ha="right")
    plt.ylabel("Normalized importance (0-1)")
    plt.title(f"Feature importance comparison (top {top_n_plot} by avg_score)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("\nCross-method feature comparison finished.")

    return results_sorted



In [ ]:
def train_models_with_top_features(
    df: pd.DataFrame,
    feature_rank: pd.DataFrame,
    target_col: str = "owners_mid",
    top_k: int = 15,
):
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataframe.")

    # 1) 依 avg_score 選出前 top_k 個特徵名稱
    top_features = feature_rank.sort_values("avg_score", ascending=False).head(top_k).index.tolist()

    print("\n=== Training models with top features ===")
    print(f"Target (y): {target_col}")
    print(f"Top {top_k} features (X):")
    print(top_features)

    # 2) 準備 X, y
    X = df[top_features].fillna(0)
    y = df[target_col].values

    # 3) 切 train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=RANDOM_STATE
    )

    # 一個小工具函式：計算並印出指標
    def evaluate_model(name, model, X_train, y_train, X_test, y_test):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        print(f"\n--- {name} ---")
        print(f"RMSE: {rmse:.2f}")
        print(f"MAE : {mae:.2f}")
        print(f"R²  : {r2:.4f}")

        return model, y_pred, (rmse, mae, r2)

    # 4) 模型二：XGBoost
    xgb_model = xgb.XGBRegressor(
        n_estimators=400,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        tree_method="hist",   # 如果有 GPU 可以改 "gpu_hist"
    )
    xgb_model, xgb_y_pred, xgb_scores = evaluate_model(
        "XGBRegressor", xgb_model, X_train, y_train, X_test, y_test
    )
    # 如果之後想拿來做預測，可以把 model 和使用的特徵一起回傳
    return {
        "features": top_features,
        "xgb_model": xgb_model,
        "xgb_scores": xgb_scores,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "xgb_y_pred": xgb_y_pred,
    }




In [ ]:
def plot_prediction_results(model_bundle, model_name: str = "xgb"):

    X_test = model_bundle["X_test"]
    y_test = model_bundle["y_test"]

    if model_name == "xgb":
        y_pred = model_bundle["xgb_y_pred"]
        title_prefix = "XGBoost"
    else:
        y_pred = model_bundle["rf_y_pred"]
        title_prefix = "RandomForest"

    # 1) 實際 vs 預測
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.4)
    max_val = max(np.max(y_test), np.max(y_pred))
    min_val = min(np.min(y_test), np.min(y_pred))
    plt.plot([min_val, max_val], [min_val, max_val], "r--", label="Ideal y = x")
    plt.xlabel("True owners_mid")
    plt.ylabel("Predicted owners_mid")
    plt.title(f"{title_prefix}: True vs Predicted")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 2) 殘差分布
    residuals = y_test - y_pred
    plt.figure(figsize=(6, 4))
    plt.hist(residuals, bins=50)
    plt.xlabel("Residual (y_true - y_pred)")
    plt.ylabel("Count")
    plt.title(f"{title_prefix}: Residual distribution")
    plt.tight_layout()
    plt.show()




In [ ]:
def analyze_market_factors(model_bundle):

    xgb_model = model_bundle["xgb_model"]
    X_train = model_bundle["X_train"]
    feature_names = X_train.columns.tolist()

    print("\n=== Market factor analysis: XGBoost feature importance ===")

    # 1) XGBoost 內建的 feature importance
    importances = xgb_model.feature_importances_
    imp_series = pd.Series(importances, index=feature_names).sort_values(ascending=False)

    print("\n[1] XGBoost feature importance (前 20 名):")
    print(imp_series.head(20))

    plt.figure(figsize=(8, max(4, len(imp_series) * 0.3)))
    imp_series.head(20).iloc[::-1].plot(kind="barh")  # 反轉一下讓最重要在最上面
    plt.xlabel("Importance")
    plt.title("XGBoost feature importance (top 20)")
    plt.tight_layout()
    plt.show()



In [ ]:
# =========================
# Main
# =========================



# 1. 讀取資料
df_raw = load_steam_dataset()

# 2. 基本 EDA
basic_eda(df_raw)

# 3. 清理資料
df_clean = clean_data(df_raw)

# 4. 特徵工程
df_feature = engineer_features(df_clean)

# 5. 特徵選擇（owners_mid 作為銷量 proxy）
results_sorted = run_feature_selection(df_feature, target_col="owners_mid", top_n_plot=len(df_feature.select_dtypes(include=[np.number]).columns)-1)
# 用排名結果訓練模型（例如用前 15 個特徵）
model_bundle = train_models_with_top_features(
    df_feature,
    results_sorted,
    target_col="owners_mid",
    top_k=15,
)
plot_prediction_results(model_bundle, model_name="xgb") 
analyze_market_factors(model_bundle)


